[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vonKleve/csci-e222-final-project/blob/master/99-comparative-analysis.ipynb)

# Comparative Analysis: Bio_ClinicalBERT vs. MedGemma-4B

## Description
This notebook conducts a comparative analysis between traditional Encoder-based architectures (BERT) and modern Decoder-based Large Language Models (Gemma) on two specialized medical tasks:
1. **Medical Named Entity Recognition (NER)** using the `MedMentions` dataset.
2. **Medical Multiple-Choice QA** using the `MedQA` dataset.

### The Models
We will evaluate the following models previously published to the Hugging Face Hub:
* **NER (Classification):** `alexd063/bio-clinicalbert-finetuned-medmentions-v1`
* **NER (Generative):** `alexd063/gemma4bit-finetuned-medmentions`
* **QA (Classification):** `alexd063/bio-clinicalbert-finetuned-medqa`
* **QA (Generative):** `alexd063/gemma4bit-finetuned-medqa`

### Architecture Paradigms
* **BERT** treats NER as *Token Classification* and QA as *Sequence Classification* (outputting logits for 4 options).
* **Gemma** treats everything as *Text-to-Text Generation*, requiring strict prompt formatting and parsing of the output string.

In [ ]:
%%capture
# !pip install --upgrade transformers datasets evaluate scikit-learn matplotlib seaborn accelerate bitsandbytes peft
!pip install --upgrade transformers bitsandbytes accelerate trl unsloth datasets

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

from datasets import load_dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForMultipleChoice,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Global Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit Quantization Config for Gemma
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Using device: {DEVICE}")

In [ ]:
import os
os.environ["TQDM_DISABLE"] = "1"

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import disable_progress_bars
disable_progress_bars()

from huggingface_hub.utils import disable_progress_bars as hub_disable_progress_bars
hub_disable_progress_bars()

---
## Part 1: Medical Named Entity Recognition (MedMentions)
We will run a qualitative side-by-side comparison. For BERT, we use the standard Hugging Face `TokenClassification` pipeline. For Gemma, we construct a prompt and parse the generative output.

In [ ]:
print("Loading MedMentions Dataset...")
ner_dataset = load_dataset("Ben10x/MedMentions-MTI881-NER", split="test")

# 1. Load BERT NER Model
print("Loading BERT NER...")
bert_ner_id = "alexd063/bio-clinicalbert-finetuned-medmentions-v1"
bert_ner_pipe = pipeline("ner", model=bert_ner_id, aggregation_strategy="simple",
                         device=0 if DEVICE=="cuda" else -1)

# 2. Load Gemma NER Model
print("Loading Gemma NER...")
gemma_ner_id = "alexd063/gemma4bit-finetuned-medmentions"
gemma_ner_tokenizer = AutoTokenizer.from_pretrained(gemma_ner_id)
gemma_ner_model = AutoModelForCausalLM.from_pretrained(
    gemma_ner_id,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
def compare_ner(sample_idx=0):
    sample = ner_dataset[sample_idx]
    text = " ".join(sample["tokens"])

    print("="*80)
    print(" INPUT TEXT:")
    print(text)
    print("="*80)

    # --- BERT INFERENCE ---
    bert_results = bert_ner_pipe(text)
    bert_entities = [f"{ent['word']} ({ent['entity_group']})" for ent in bert_results]

    print("\n[Bio_ClinicalBERT Predictions]:")
    if bert_entities:
        print(", ".join(bert_entities))
    else:
        print("No entities found.")

    # --- GEMMA INFERENCE ---
    instruction = "Extract medical entities from the following text and label each token with its NER tag."
    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{text}\n\n"
        f"### Response:\n"
    )

    inputs = gemma_ner_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gemma_ner_model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=False
        )

    decoded = gemma_ner_tokenizer.decode(outputs[0], skip_special_tokens=True)
    gemma_response = decoded.split("### Response:\n")[-1].strip()

    print("\n[MedGemma-4B Predictions]:")
    print(gemma_response)
    print("="*80 + "\n")

# Test on 3 samples
for i in range(3):
    compare_ner(sample_idx=i)

---
## Part 2: Medical Question Answering (MedQA)
We will evaluate both models on a subset of the MedQA test split to calculate quantitative accuracy.

* **BERT Pipeline:** We manually feed `(Question, Option)` pairs into the model and apply softmax to the 4 output logits.
* **Gemma Pipeline:** We format the question as a prompt and restrict generation to exactly 2 tokens (the answer letter + EOS).

In [ ]:
print("Loading MedQA Dataset...")
qa_dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
qa_subset = qa_dataset.select(range(200)) # Evaluate on 200 samples for time

# 1. Load BERT QA Model
print("Loading BERT QA...")
bert_qa_id = "alexd063/bio-clinicalbert-finetuned-medqa"
bert_qa_tokenizer = AutoTokenizer.from_pretrained(bert_qa_id)
bert_qa_model = AutoModelForMultipleChoice.from_pretrained(bert_qa_id).to(DEVICE)
bert_qa_model.eval()

In [ ]:
# 2. Load Gemma QA Model
print("Loading Gemma QA...")
gemma_qa_id = "alexd063/gemma4bit-finetuned-medqa"
gemma_qa_tokenizer = AutoTokenizer.from_pretrained(gemma_qa_id)
gemma_qa_model = AutoModelForCausalLM.from_pretrained(
    gemma_qa_id,
    load_in_8bit=False,
    device_map=DEVICE
)
gemma_qa_model.eval()
print("Models loaded successfully.")

In [ ]:
true_labels = []
bert_predictions = []
gemma_predictions = []

print("Running Inference on QA Subset...")

for sample in tqdm(qa_subset):
    question = sample["question"]
    opts = sample["options"]
    answer_idx = sample["answer_idx"].upper()
    true_labels.append(answer_idx)

    # -------------------------
    # 1. BERT Inference
    # -------------------------
    choices = [opts.get('A',''), opts.get('B',''), opts.get('C',''), opts.get('D','')]
    # Format for MultipleChoice: [[Q, A], [Q, B], [Q, C], [Q, D]]
    bert_inputs = bert_qa_tokenizer(
        [[question, c] for c in choices],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    # Add batch dimension: (1, 4, seq_length)
    bert_inputs = {k: v.unsqueeze(0).to(DEVICE) for k, v in bert_inputs.items()}

    with torch.no_grad():
        bert_outputs = bert_qa_model(**bert_inputs)
        predicted_class = torch.argmax(bert_outputs.logits, dim=1).item()

    # Map index 0-3 back to A-D
    bert_predictions.append(["A", "B", "C", "D"][predicted_class])

    # -------------------------
    # 2. Gemma Inference
    # -------------------------
    instruction = "Answer the following multiple-choice medical question. Provide only the letter of the correct option (A, B, C, or D)."
    options_text = f"A: {opts.get('A', '')}\nB: {opts.get('B', '')}\nC: {opts.get('C', '')}\nD: {opts.get('D', '')}"

    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Question:\n{question}\n\n"
        f"### Options:\n{options_text}\n\n"
        f"### Answer:\n"
    )

    gemma_inputs = gemma_qa_tokenizer([prompt], return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        gemma_outputs = gemma_qa_model.generate(
            **gemma_inputs,
            max_new_tokens=2,
            temperature=0.1,
            pad_token_id=gemma_qa_tokenizer.eos_token_id
        )

    decoded = gemma_qa_tokenizer.batch_decode(gemma_outputs, skip_special_tokens=True)[0]
    gemma_pred = decoded.split("### Answer:\n")[-1].strip()[:1].upper()

    # Fallback if generation failed to produce A/B/C/D
    if gemma_pred not in ["A", "B", "C", "D"]:
        gemma_pred = "A"

    gemma_predictions.append(gemma_pred)

print("\nInference Complete.")